# Alaska glacier coverage by OPERA RTC-S1 bursts
One 12-day acquisition cycle is deduplicated by `operaBurstID`, intersected with RGI 7.0 glaciers clipped to the State of Alaska, and mapped. The exact live count can shift slightly as catalog products or boundary versions change; round it up for budgeting.

In [ ]:
import asf_search as asf, geopandas as gpd, matplotlib.pyplot as plt
from shapely.geometry import box
RGI = 'Glaciers/RGI2000-v7.0-G-01_alaska/RGI2000-v7.0-G-01_alaska.shp'
CENSUS = 'https://www2.census.gov/geo/tiger/GENZ2025/shp/cb_2025_us_state_20m.zip'

In [ ]:
ak = gpd.read_file(CENSUS).query("STUSPS == 'AK'").to_crs(4326)
glaciers = gpd.overlay(gpd.read_file(RGI).to_crs(4326), ak[['geometry']], how='intersection')
results = asf.geo_search(dataset='OPERA-S1', processingLevel='RTC', intersectsWith=box(-177, 52, -126, 72).wkt, start='2024-07-01', end='2024-07-13')
bursts = gpd.GeoDataFrame.from_features(results.geojson(), crs=4326).sort_values('startTime').drop_duplicates('operaBurstID')
hits = gpd.sjoin(bursts, glaciers[['geometry']], predicate='intersects').drop_duplicates('operaBurstID')
len(hits)

In [ ]:
crs = 3338; ax = ak.to_crs(crs).plot(figsize=(12, 8), color='0.92', edgecolor='0.35')
glaciers.to_crs(crs).plot(ax=ax, color='crimson', linewidth=0, alpha=.75)
hits.to_crs(crs).boundary.plot(ax=ax, color='royalblue', linewidth=.35, alpha=.65)
ax.set_title(f'{len(hits)} unique OPERA RTC-S1 burst IDs intersect Alaska glaciers → budget for ~400'); ax.set_axis_off()